<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-Embeddings-Semantic-Retrieval-Platform/blob/main/Enterprise_Embeddings_Semantic_Retrieval_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime
import random
import json
import re

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE_PATH = Path('/content/enterprise_embeddings_semantic_retrieval_platform')
RAW_PATH = BASE_PATH / 'data' / 'raw_documents'
CHUNK_PATH = BASE_PATH / 'data' / 'chunked_documents'
VECTOR_READY_PATH = BASE_PATH / 'data' / 'vector_ready'
OUTPUT_PATH = BASE_PATH / 'outputs'
CONFIG_PATH = BASE_PATH / 'config'

for p in [RAW_PATH, CHUNK_PATH, VECTOR_READY_PATH, OUTPUT_PATH, CONFIG_PATH]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)

print('Project folders created successfully.')
print(BASE_PATH)

Project folders created successfully.
/content/enterprise_embeddings_semantic_retrieval_platform


In [2]:
retrieval_schema = {
    'document_domains': ['engineering', 'quality', 'manufacturing'],
    'metadata_fields': [
        'document_id',
        'document_type',
        'domain',
        'plant_code',
        'supplier_name',
        'equipment_id',
        'document_date',
        'chunk_id'
    ],
    'pipeline_steps': [
        'document_preparation',
        'chunking',
        'metadata_enrichment',
        'embedding_generation',
        'semantic_retrieval'
    ]
}

with open(CONFIG_PATH / 'retrieval_schema.json', 'w') as f:
    json.dump(retrieval_schema, f, indent=2)

print(json.dumps(retrieval_schema, indent=2))

{
  "document_domains": [
    "engineering",
    "quality",
    "manufacturing"
  ],
  "metadata_fields": [
    "document_id",
    "document_type",
    "domain",
    "plant_code",
    "supplier_name",
    "equipment_id",
    "document_date",
    "chunk_id"
  ],
  "pipeline_steps": [
    "document_preparation",
    "chunking",
    "metadata_enrichment",
    "embedding_generation",
    "semantic_retrieval"
  ]
}


In [3]:
documents = []

supplier_names = ['Magna', 'Bosch', 'Denso', 'Lear', 'Aptiv']
plants = ['PLT100', 'PLT200', 'PLT300', 'PLT400']
equipment_ids = ['EQP1001', 'EQP1002', 'EQP1003', 'EQP1004']
issue_types = ['weld defect', 'paint inconsistency', 'sensor failure', 'material shortage', 'assembly variance']

for i in range(1, 301):
    domain = random.choice(retrieval_schema['document_domains'])
    document_type = random.choice(['inspection_report', 'maintenance_note', 'supplier_record', 'engineering_change'])
    plant = random.choice(plants)
    supplier = random.choice(supplier_names)
    equipment = random.choice(equipment_ids)
    issue = random.choice(issue_types)
    doc_date = f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"

    text = (
        f"Document ID DOC{i:05d}. Domain {domain}. Document type {document_type}. Plant {plant}. "
        f"Supplier {supplier}. Equipment {equipment}. Issue {issue}. Document date {doc_date}. "
        f"This document explains operational impact, engineering context, quality observations, "
        f"recommended action, and downstream business usage for plant operations and review."
    )

    documents.append({
        'document_id': f'DOC{i:05d}',
        'domain': domain,
        'document_type': document_type,
        'document_text': text
    })

documents_df = pd.DataFrame(documents)
documents_df.to_csv(RAW_PATH / 'enterprise_documents.csv', index=False)
print(documents_df.head())

  document_id         domain      document_type  \
0    DOC00001  manufacturing  inspection_report   
1    DOC00002    engineering  inspection_report   
2    DOC00003  manufacturing  inspection_report   
3    DOC00004        quality  inspection_report   
4    DOC00005        quality  inspection_report   

                                       document_text  
0  Document ID DOC00001. Domain manufacturing. Do...  
1  Document ID DOC00002. Domain engineering. Docu...  
2  Document ID DOC00003. Domain manufacturing. Do...  
3  Document ID DOC00004. Domain quality. Document...  
4  Document ID DOC00005. Domain quality. Document...  


In [5]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_field(pattern, text):
    match = re.search(pattern, text, re.IGNORECASE)
    return match.group(1) if match else None


documents_df = pd.read_csv(RAW_PATH / 'enterprise_documents.csv')
documents_df['normalized_text'] = documents_df['document_text'].apply(normalize_text)
documents_df['plant_code'] = documents_df['document_text'].apply(lambda x: extract_field(r'Plant\s+(PLT\d+)', x))
documents_df['supplier_name'] = documents_df['document_text'].apply(lambda x: extract_field(r'Supplier\s+(Magna|Bosch|Denso|Lear|Aptiv)', x))
documents_df['equipment_id'] = documents_df['document_text'].apply(lambda x: extract_field(r'Equipment\s+(EQP\d+)', x))
documents_df['document_date'] = documents_df['document_text'].apply(lambda x: extract_field(r'(\d{4}-\d{2}-\d{2})', x))

documents_df.to_csv(VECTOR_READY_PATH / 'document_metadata_base.csv', index=False)
print(documents_df.head())

  document_id         domain       document_type  \
0    DOC00001  manufacturing  engineering_change   
1    DOC00002        quality    maintenance_note   
2    DOC00003    engineering   inspection_report   
3    DOC00004  manufacturing   inspection_report   
4    DOC00005    engineering   inspection_report   

                                       document_text  \
0  Document ID DOC00001. Domain manufacturing. Do...   
1  Document ID DOC00002. Domain quality. Document...   
2  Document ID DOC00003. Domain engineering. Docu...   
3  Document ID DOC00004. Domain manufacturing. Do...   
4  Document ID DOC00005. Domain engineering. Docu...   

                                     normalized_text plant_code supplier_name  \
0  document id doc00001. domain manufacturing. do...     PLT400         Bosch   
1  document id doc00002. domain quality. document...     PLT100         Magna   
2  document id doc00003. domain engineering. docu...     PLT100         Aptiv   
3  document id doc00004. d

In [6]:
def simple_chunk_text(text, chunk_size=25):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

chunk_rows = []
for _, row in documents_df.iterrows():
    chunks = simple_chunk_text(row['normalized_text'], chunk_size=25)
    for idx, chunk in enumerate(chunks, start=1):
        chunk_rows.append({
            'document_id': row['document_id'],
            'chunk_id': f"{row['document_id']}_CHUNK_{idx}",
            'domain': row['domain'],
            'document_type': row['document_type'],
            'plant_code': row['plant_code'],
            'supplier_name': row['supplier_name'],
            'equipment_id': row['equipment_id'],
            'document_date': row['document_date'],
            'chunk_text': chunk
        })

chunk_df = pd.DataFrame(chunk_rows)
chunk_df.to_csv(CHUNK_PATH / 'chunked_documents.csv', index=False)
print(chunk_df.head())
print('Total chunks:', len(chunk_df))

  document_id          chunk_id         domain       document_type plant_code  \
0    DOC00001  DOC00001_CHUNK_1  manufacturing  engineering_change     PLT400   
1    DOC00001  DOC00001_CHUNK_2  manufacturing  engineering_change     PLT400   
2    DOC00002  DOC00002_CHUNK_1        quality    maintenance_note     PLT100   
3    DOC00002  DOC00002_CHUNK_2        quality    maintenance_note     PLT100   
4    DOC00003  DOC00003_CHUNK_1    engineering   inspection_report     PLT100   

  supplier_name equipment_id document_date  \
0         Bosch      EQP1002    2024-06-05   
1         Bosch      EQP1002    2024-06-05   
2         Magna      EQP1004    2024-01-08   
3         Magna      EQP1004    2024-01-08   
4         Aptiv      EQP1001    2024-10-22   

                                          chunk_text  
0  document id doc00001. domain manufacturing. do...  
1  engineering context, quality observations, rec...  
2  document id doc00002. domain quality. document...  
3  engineering c

In [7]:
chunk_df['vector_ready_text'] = (
    'Domain: ' + chunk_df['domain'] + ' | ' +
    'DocumentType: ' + chunk_df['document_type'] + ' | ' +
    'Plant: ' + chunk_df['plant_code'].fillna('UNKNOWN') + ' | ' +
    'Supplier: ' + chunk_df['supplier_name'].fillna('UNKNOWN') + ' | ' +
    'Equipment: ' + chunk_df['equipment_id'].fillna('UNKNOWN') + ' | ' +
    'Date: ' + chunk_df['document_date'].fillna('UNKNOWN') + ' | ' +
    'Text: ' + chunk_df['chunk_text']
)

chunk_df.to_csv(VECTOR_READY_PATH / 'vector_ready_records.csv', index=False)
chunk_df.to_csv(OUTPUT_PATH / 'vector_records.csv', index=False)
print(chunk_df[['chunk_id', 'vector_ready_text']].head())

           chunk_id                                  vector_ready_text
0  DOC00001_CHUNK_1  Domain: manufacturing | DocumentType: engineer...
1  DOC00001_CHUNK_2  Domain: manufacturing | DocumentType: engineer...
2  DOC00002_CHUNK_1  Domain: quality | DocumentType: maintenance_no...
3  DOC00002_CHUNK_2  Domain: quality | DocumentType: maintenance_no...
4  DOC00003_CHUNK_1  Domain: engineering | DocumentType: inspection...


In [8]:
vectorizer = TfidfVectorizer(stop_words='english')
embedding_matrix = vectorizer.fit_transform(chunk_df['vector_ready_text'])

print('Embedding matrix shape:', embedding_matrix.shape)

Embedding matrix shape: (600, 382)


In [9]:
query = 'engineering document about sensor failure on equipment EQP1002 in plant PLT200'
query_vector = vectorizer.transform([query])
similarity_scores = cosine_similarity(query_vector, embedding_matrix).flatten()

top_indices = similarity_scores.argsort()[-10:][::-1]
search_results_df = chunk_df.iloc[top_indices].copy()
search_results_df['similarity_score'] = similarity_scores[top_indices]
search_results_df = search_results_df[[
    'chunk_id', 'document_id', 'domain', 'document_type', 'plant_code',
    'supplier_name', 'equipment_id', 'document_date', 'chunk_text', 'similarity_score'
]]

search_results_df.to_csv(OUTPUT_PATH / 'semantic_search_results.csv', index=False)
search_results_df.head(10)

,chunk_id,document_id,domain,document_type,plant_code,supplier_name,equipment_id,document_date,chunk_text,similarity_score
438,DOC00220_CHUNK_1,DOC00220,engineering,engineering_change,PLT200,Aptiv,EQP1002,2024-09-07,document id doc00220. domain engineering. docu...,0.553501
20,DOC00011_CHUNK_1,DOC00011,engineering,engineering_change,PLT200,Denso,EQP1002,2024-11-03,document id doc00011. domain engineering. docu...,0.545139
576,DOC00289_CHUNK_1,DOC00289,engineering,supplier_record,PLT200,Aptiv,EQP1002,2024-12-21,document id doc00289. domain engineering. docu...,0.522913
104,DOC00053_CHUNK_1,DOC00053,quality,engineering_change,PLT200,Denso,EQP1002,2024-11-05,document id doc00053. domain quality. document...,0.514800
486,DOC00244_CHUNK_1,DOC00244,engineering,engineering_change,PLT200,Lear,EQP1002,2024-02-25,document id doc00244. domain engineering. docu...,0.513283
222,DOC00112_CHUNK_1,DOC00112,manufacturing,supplier_record,PLT200,Denso,EQP1002,2024-12-04,document id doc00112. domain manufacturing. do...,0.500833
166,DOC00084_CHUNK_1,DOC00084,manufacturing,maintenance_note,PLT200,Bosch,EQP1002,2024-04-25,document id doc00084. domain manufacturing. do...,0.472258
326,DOC00164_CHUNK_1,DOC00164,engineering,engineering_change,PLT200,Magna,EQP1001,2024-04-07,document id doc00164. domain engineering. docu...,0.457126
174,DOC00088_CHUNK_1,DOC00088,engineering,engineering_change,PLT200,Lear,EQP1003,2024-05-11,document id doc00088. domain engineering. docu...,0.456155
162,DOC00082_CHUNK_1,DOC00082,engineering,supplier_record,PLT300,Aptiv,EQP1002,2024-12-10,document id doc00082. domain engineering. docu...,0.444932


In [10]:
chunk_metadata_df = chunk_df[[
    'document_id', 'chunk_id', 'domain', 'document_type',
    'plant_code', 'supplier_name', 'equipment_id', 'document_date'
]].copy()

chunk_metadata_df.to_csv(OUTPUT_PATH / 'chunk_metadata.csv', index=False)
chunk_metadata_df.head()

,document_id,chunk_id,domain,document_type,plant_code,supplier_name,equipment_id,document_date
0,DOC00001,DOC00001_CHUNK_1,manufacturing,engineering_change,PLT400,Bosch,EQP1002,2024-06-05
1,DOC00001,DOC00001_CHUNK_2,manufacturing,engineering_change,PLT400,Bosch,EQP1002,2024-06-05
2,DOC00002,DOC00002_CHUNK_1,quality,maintenance_note,PLT100,Magna,EQP1004,2024-01-08
3,DOC00002,DOC00002_CHUNK_2,quality,maintenance_note,PLT100,Magna,EQP1004,2024-01-08
4,DOC00003,DOC00003_CHUNK_1,engineering,inspection_report,PLT100,Aptiv,EQP1001,2024-10-22


In [11]:
summary = {
    'generated_at': datetime.now().isoformat(),
    'raw_document_count': len(documents_df),
    'chunk_count': len(chunk_df),
    'vector_ready_record_count': len(chunk_df),
    'top_query': query,
    'top_result_document_id': search_results_df.iloc[0]['document_id'] if len(search_results_df) > 0 else None
}

with open(OUTPUT_PATH / 'project_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "generated_at": "2026-03-17T17:43:20.590330",
  "raw_document_count": 300,
  "chunk_count": 600,
  "vector_ready_record_count": 600,
  "top_query": "engineering document about sensor failure on equipment EQP1002 in plant PLT200",
  "top_result_document_id": "DOC00220"
}
